# Sky model and bandpass templates

Two kinds of signal are not described by one map shared by all the detectors,
and bias the calibration (see the calibration notebook):

- **structure below the pixel size**: the samples see the sky at their exact
  pointing, the map has one value per pixel. Given as alm, a `sky_model` is
  interpolated at the pointing (`ducc0.sht.synthesis_general`) and subtracted;
  its binned map is put back, so the maps are still those of the whole sky.
- **bandpass mismatch**: detectors of the same channel see the foregrounds
  through slightly different bands. `bandpass_templates` get one amplitude per
  detector (summing to zero: the common part is the map), fitted with the maps
  and the gains. With the derivative of the foreground with respect to
  frequency, $F_i \simeq F + \Delta\nu_i\,\partial F/\partial\nu$, the
  amplitude is the shift of the band $\Delta\nu_i$.

Here: the data of the calibration notebook (4 detectors, 1/f noise, gains with
1% offsets and a 3% drift), with a CMB and a dust foreground at 140 GHz: 3 mK
in a Galactic band with structure at 1°, 10% polarized, spectral index
$\beta = 1.5 \pm 0.2$ varying across the sky, seen by each detector through its
band, shifted by $\Delta\nu_i$ = +1.5, −1.0, +0.8, −1.3 GHz. The sky is
simulated at nside 128, the maps are at nside 16.

In [1]:
import dataclasses
import ducc0
import healpy as hp
import numpy as np
from pysanepic import CalibrationParameters, DetectorData, GLSParameters, OneOverFNoise, Pointing, make_maps

nside, fs, n = 16, 10.0, 240_000
hi, lmax = 128, 256
rng = np.random.default_rng(3)
npix, nhi = 12 * nside**2, 12 * hi**2
z, zh = hp.pix2vec(nside, np.arange(npix))[2], hp.pix2vec(hi, np.arange(nhi))[2]
cmb_alm = hp.map2alm(hp.smoothing(rng.normal(0, 1e-4, (3, nhi)), fwhm=np.radians(6)), lmax=lmax)
# dust: amplitude A (3 mK in a band, structure at 1 degree), 10% polarized, spectral index beta(p)
A = np.where(np.abs(zh) < 0.2, 3e-3, 2e-5) * (1 + 0.3 * hp.smoothing(rng.normal(0, 3, nhi), fwhm=np.radians(1)))
A = np.maximum(A, 0)
pa = hp.smoothing(rng.normal(0, 1, nhi), fwhm=np.radians(3)) * np.pi
dust = np.array([A, 0.1 * A * np.cos(2 * pa), 0.1 * A * np.sin(2 * pa)])
beta = 1.5 + 0.2 * hp.smoothing(rng.normal(0, 3, nhi), fwhm=np.radians(5))
nu0 = 140.0
shift = np.array([1.5, -1.0, 0.8, -1.3])  # GHz: band centre of each detector minus nu0
alm = lambda m: hp.map2alm(m, lmax=lmax)
F, F1, F2 = alm(dust), alm(dust * beta / nu0), alm(dust * beta * (beta - 1) / nu0**2)
noise = OneOverFNoise(net_ukrts=35.0, fknee_hz=0.05, alpha=1.0, fmin_hz=1e-4)
t = np.arange(n)
lam = 2 * np.pi * t / n + rng.uniform(0, 2 * np.pi)
spin = 2 * np.pi * t / 3000.0
a = np.array([np.cos(lam), np.sin(lam), np.zeros(n)])
e2 = np.cross(a.T, [0, 0, 1]).T
vec = np.cos(1.3) * a + np.sin(1.3) * (np.cos(spin) * np.array([[0], [0], [1]]) + np.sin(spin) * e2)
theta, phi = np.arccos(np.clip(vec[2], -1, 1)), np.arctan2(vec[1], vec[0]) % (2 * np.pi)
psi = (spin + lam) % (2 * np.pi) - np.pi
solar = 3.36e-3 * np.array([0.3, -0.5, 0.8]) / np.linalg.norm([0.3, -0.5, 0.8])
dipole = solar @ vec + np.sum(2.7e-4 * np.array([-np.sin(lam), np.cos(lam), 0 * t]) * vec, axis=0)
loc = np.stack([theta, phi], axis=1)


def interp(alm, ang):
    i = ducc0.sht.synthesis_general(alm=alm[:1], spin=0, lmax=lmax, loc=loc, epsilon=1e-10)[0]
    q, u = ducc0.sht.synthesis_general(alm=alm[1:], spin=2, lmax=lmax, loc=loc, epsilon=1e-10)
    return i + q * np.cos(2 * ang) + u * np.sin(2 * ang)


def draw(n):
    return np.fft.irfft(np.fft.rfft(rng.normal(size=n)) / np.sqrt(noise.weights(n, fs)), n=n)


data, true_gain = [], []
for i, pol_angle in enumerate(np.radians([0, 90, 45, 135])):
    ang = psi + pol_angle
    dust_i = alm(dust * ((nu0 + shift[i]) / nu0) ** beta)  # the dust through the band of detector i
    signal = interp(cmb_alm, ang) + interp(dust_i, ang) + dipole
    g = 1 + 0.01 * rng.normal() + 0.03 * np.cumsum(rng.normal(size=n)) / np.sqrt(n)
    true_gain.append(g)
    data.append(DetectorData(tod=g * signal + draw(n), pointing=Pointing(theta=theta, phi=phi, psi=psi, coordinates="E"),
                             sampling_rate_hz=fs, noise=noise, pol_angle_rad=pol_angle, dipole=dipole, name=f"d{i}"))
params = GLSParameters(chunk_s=2400.0, tol=1e-12)
galaxy = (np.abs(z) > 0.3).astype(float)  # a mask: 1 = used, 0 on the Galactic band

The reference: maps from the data calibrated with the true gains, with the
exact model and the bandpass fitted. The rest is compared to it outside the
Galactic band, in µK; gains as in the calibration notebook (rms of
$g/g_\text{true} - 1$ and its mean, the absolute calibration).

In [2]:
def report(name, r):
    ok = (r.maps[1] != hp.UNSEEN) & (np.abs(z) > 0.3)
    e = (r.maps - ref.maps)[:, ok] * 1e6
    gerr = np.array([g[t // 12000] / g0 - 1 for g, g0 in zip(r.gains, true_gain)])
    line = f"{name:30s} I/Q/U {e[0].std():5.2f} / {e[1].std():5.2f} / {e[2].std():5.2f} uK   gains rms {np.sqrt(np.mean(gerr**2)):.1e}, absolute {gerr.mean():+.1e}"
    for k, b in (r.bandpass or {}).items():
        line += f"\n{'':30s} {k}: " + ", ".join(f"{v:+.3g}" for v in b.values())
    print(line)


model = {"cmb": cmb_alm, "fg": F}
calibrated = [dataclasses.replace(d, tod=d.tod / g - dipole, dipole=None) for d, g in zip(data, true_gain)]
ref = make_maps(calibrated, nside, coordinates="E", params=params, sky_model=model, bandpass_templates={"fg_deriv1": F1})
cal = CalibrationParameters(gain_period_s=1200.0)
print("band shifts - their mean [GHz]:", shift - shift.mean())

band shifts - their mean [GHz]: [ 1.5 -1.   0.8 -1.3]


## Structure below the pixel size

Without a model the Galaxy has to be masked for the calibration, and its
structure below the pixel size still leaks into the I map through the 1/f
weights. With the model as alm, no mask is needed and the I map is right; Q
and U are still off, as the gains are biased by the bandpass mismatch.

In [3]:
report("no model, Galactic mask", make_maps(data, nside, coordinates="E", params=params,
                                            calibration=CalibrationParameters(gain_period_s=1200.0, mask=galaxy)))
report("sky model (alm), no mask", make_maps(data, nside, coordinates="E", params=params, calibration=cal, sky_model=model))

no model, Galactic mask        I/Q/U 19.27 /  5.16 /  5.06 uK   gains rms 4.7e-03, absolute -3.1e-03


sky model (alm), no mask       I/Q/U  0.93 /  3.80 /  3.68 uK   gains rms 6.3e-03, absolute +1.6e-04


## Bandpass mismatch

The dust seen by each detector differs by a few percent: without a template
the difference goes into the gains and, between the two detectors of a pair,
into Q and U. A template per detector removes it: the foreground itself
(`"fg"`, one scale per detector) or its first derivative (`"fg_deriv1"`, whose
amplitudes are the band shifts in GHz).

In [4]:
for name, templates in (("bandpass: fg", {"fg": F}), ("bandpass: fg_deriv1", {"fg_deriv1": F1}),
                        ("bandpass: fg_deriv1 + fg_deriv2", {"fg_deriv1": F1, "fg_deriv2": F2})):
    report(name, make_maps(data, nside, coordinates="E", params=params, calibration=cal,
                           sky_model=model, bandpass_templates=templates))

bandpass: fg                   I/Q/U  0.93 /  1.26 /  1.31 uK   gains rms 3.3e-03, absolute +1.9e-04
                               fg: +0.0155, -0.00911, +0.00816, -0.0145


bandpass: fg_deriv1            I/Q/U  0.93 /  1.26 /  1.31 uK   gains rms 3.3e-03, absolute +1.9e-04
                               fg_deriv1: +1.44, -0.847, +0.756, -1.35


bandpass: fg_deriv1 + fg_deriv2 I/Q/U  0.93 /  1.26 /  1.31 uK   gains rms 3.3e-03, absolute +1.9e-04
                               fg_deriv1: +0.959, -1.14, +1.33, -1.15
                               fg_deriv2: +134, +80.8, -159, -55.3


The first derivative recovers the band shifts within the noise, and Q/U
drop from 3.8 to 1.3 µK; `"fg"` alone does as well here. The second derivative
is nearly proportional to the first ($\partial^2F/\partial\nu^2 =
(\beta - 1)/\nu\;\partial F/\partial\nu$, with $\beta$ varying little), so the
two amplitudes are not separable: the maps do not change, but the amplitudes
are meaningless. Pass templates that differ on the sky.

## Masks

`CalibrationParameters.mask` can be one map, for all the fits, or
`{"gain": ..., "bandpass": ...}`, e.g. `{"gain": galaxy * sources, "bandpass":
sources}`: the bandpass amplitudes are fitted on the Galaxy, where the
foreground is bright, and point sources missing from the model are left out
of everything. Samples left out of the gain fit only are calibrated with the
gains of the previous iteration, so the iterations converge slower; with a
good model, as here, no mask is needed.